In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from keras.models import Model
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers

In [ ]:
# Get a list of all available physical GPUs
gpus = tf.config.experimental.list_physical_devices('GPU')

if gpus:
    try:
        # Enable memory growth for each detected GPU
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth enabled for GPU")
    except RuntimeError as e:
        # Memory growth must be set before initializing the TensorFlow runtime
        print(f"Error setting memory growth: {e}")

# Load Dataset

In [ ]:
DATASET_PATH = 'Brain Tumor Segmentation Dataset/'

In [ ]:
image_base_path = os.path.join(DATASET_PATH, 'image')
image_base_path

In [ ]:
mask_base_path = os.path.join(DATASET_PATH, 'mask')
mask_base_path

In [ ]:
class_dirs = [d for d in os.listdir(image_base_path) if os.path.isdir(os.path.join(image_base_path, d))]
class_dirs

In [ ]:
all_image_paths = []
all_mask_paths = []
all_labels = []

for i in range(len(class_dirs)):
    class_dir = class_dirs[i]  

    image_folder = os.path.join(image_base_path, class_dir)
    mask_folder = os.path.join(mask_base_path, class_dir)

    label = int(class_dir)

    for file_name in os.listdir(image_folder):
        if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
            img_path = os.path.join(image_folder, file_name)
            
            base_name, extension = os.path.splitext(file_name)
            mask_filename = f"{base_name}_m{extension}"
            mask_path = os.path.join(mask_folder, mask_filename)
            
            if os.path.exists(mask_path):
                all_image_paths.append(img_path)
                all_mask_paths.append(mask_path)
                all_labels.append(label)

In [ ]:
all_image_paths[10]

In [ ]:
all_mask_paths[10]

In [ ]:
all_labels[10]

In [ ]:
len(all_image_paths)

In [ ]:
train_images, val_images, train_masks, val_masks, train_labels, val_labels = train_test_split(
    all_image_paths, all_mask_paths, all_labels, test_size=0.1, random_state=42, stratify=all_labels
)

In [ ]:
len(train_images)

In [ ]:
len(val_images)

# Masks Pixel Check

In [ ]:
problematic_masks = []
all_unique_values = set()

for mask_path in all_mask_paths:
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    unique_values = np.unique(mask)
    
    all_unique_values.update(unique_values)
    
    extra_values = set(unique_values) - {0, 255}
    
    if extra_values:
        problematic_masks.append((mask_path, unique_values))


print(f"Global unique pixel values found in all masks: {sorted(list(all_unique_values))}")

In [ ]:
all_pixels_list = []

for mask_path in all_mask_paths:
    mask_image = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    all_pixels_list.append(mask_image.ravel())

all_pixel_values = np.concatenate(all_pixels_list)

plt.figure(figsize=(12, 7))
plt.hist(all_pixel_values, bins=256, range=[0, 256], color='blue')
plt.title('Histogram of Pixel Values for ALL Masks in the Dataset')
plt.xlabel('Pixel Value (0=Black, 255=White)')
plt.ylabel('Total Number of Pixels (Frequency)')
plt.grid(True, alpha=0.5)

In [ ]:
IMG_SIZE = (256, 256)
BATCH_SIZE = 8

In [ ]:
def load_image_and_mask(image_path, mask_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)


    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.where(mask > 128, tf.cast(label, tf.uint8), tf.cast(0, tf.uint8))
    mask = tf.image.resize(mask, IMG_SIZE, method='nearest')

    return img, mask

In [ ]:
def augment_photometric(image, mask):
  
    image = tf.image.random_brightness(image, max_delta=0.05)
    image = tf.image.random_contrast(image, lower=0.95, upper=1.05)
    image = tf.clip_by_value(image, 0.0, 255.0) 
    
    return image, mask

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input

def final_preprocess_for_resnet(image, mask):
    image = preprocess_input(image)
    return image, mask

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_masks, train_labels))

train_dataset = train_dataset.map(load_image_and_mask, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.map(augment_photometric, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.map(final_preprocess_for_resnet, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=1000).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
val_dataset = tf.data.Dataset.from_tensor_slices((val_images, val_masks, val_labels))
val_dataset = val_dataset.map(load_image_and_mask, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.map(final_preprocess_for_resnet, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
print(f"Train dataset: {train_dataset}")
print(f"Validation dataset: {val_dataset}")

In [ ]:
for images_batch, masks_batch in train_dataset.take(1):
    for i in range(3):
        image = images_batch[i]
        print(f"Shape: {image.shape}")
        print(f"Dtype: {image.dtype}")
        print(f"Min value: {np.min(image):.4f}")
        print(f"Max value: {np.max(image):.4f}")
        print(f"Mean value: {np.mean(image):.4f}")
        print("-" * 25)

# Imbalance Data

In [ ]:
class_counts = {0: 0, 1: 0, 2: 0, 3: 0}
total_pixels = 0

for images_batch, masks_batch in train_dataset:
    for mask in masks_batch:
        mask_np = mask.numpy()
        total_pixels += mask_np.size
        unique, counts = np.unique(mask_np, return_counts=True)
        for u, c in zip(unique, counts):
            if u in class_counts:
                class_counts[u] += c

In [ ]:
total_pixels

In [ ]:
class_counts

# Median Frequency Balancing 

In [ ]:
class_frequencies = {cls: count / total_pixels for cls, count in class_counts.items() if count > 0}

frequencies = list(class_frequencies.values())
median_frequency = np.median(frequencies)

mfb_weights = {}
for cls, freq in class_frequencies.items():
    mfb_weights[cls] = median_frequency / freq

for cls in class_counts:
    if cls not in mfb_weights:
        mfb_weights[cls] = 0.0

print(mfb_weights)

# Modeling

In [ ]:
from tensorflow.keras.layers import Conv2D, BatchNormalization, ReLU, Add, Input, UpSampling2D, Concatenate, AveragePooling2D, GlobalAveragePooling2D

In [ ]:
def convolution_block(
    block_input,
    num_filters=256,
    kernel_size=3,
    dilation_rate=1,
    padding="same",
    use_bias=True,
):
    x = Conv2D(
        num_filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding="same",
        use_bias=use_bias,
    )(block_input)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    return x

In [ ]:
# GlobalAveragePooling2D >>>> output shape: (None, 2048)
# AveragePooling2D >>> output shape: (None, 1, 1, 2048)

def AtrousSpatialPyramidPooling(aspp_input):
    dims = aspp_input.shape
    x = AveragePooling2D(pool_size=(dims[1], dims[2]))(aspp_input)
    out_pool = UpSampling2D(size=(dims[1], dims[2]), interpolation="bilinear")(x)

    out_1 = convolution_block(aspp_input, kernel_size=1, dilation_rate=1)
    out_6 = convolution_block(aspp_input, kernel_size=3, dilation_rate=6)
    out_12 = convolution_block(aspp_input, kernel_size=3, dilation_rate=12)
    out_18 = convolution_block(aspp_input, kernel_size=3, dilation_rate=18)

    x = Concatenate(axis=-1)([out_pool, out_1, out_6, out_12, out_18])
    output = convolution_block(x, kernel_size=1)
    return output

In [ ]:
def DeeplabV3_OS16(image_size, num_classes):
  
    model_input = keras.Input(shape=(image_size, image_size, 3))
    
    weights_path = './resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5'
    base_model = keras.applications.ResNet50(weights=weights_path, include_top=False, input_tensor=model_input)
    
    base_model.trainable = False
    
    x = base_model.get_layer("conv4_block6_out").output

    low_level_features = base_model.get_layer("conv2_block3_out").output

    for i in range(1, 4):
        shortcut = x
        
        if i == 1:
            # 1x1 conv
            x = Conv2D(512, (1, 1), strides=(1, 1), padding="same", name=f"custom_conv5_block{i}_1_conv")(x)
            x = BatchNormalization(name=f"custom_conv5_block{i}_1_bn")(x)
            x = ReLU(name=f"custom_conv5_block{i}_1_relu")(x)
            
            # 3x3 atrous conv + dilation_rate=2
            x = Conv2D(512, (3, 3), strides=(1, 1), padding="same", dilation_rate=2, name=f"custom_conv5_block{i}_2_conv")(x)
            x = BatchNormalization(name=f"custom_conv5_block{i}_2_bn")(x)
            x = ReLU(name=f"custom_conv5_block{i}_2_relu")(x)

            # 1x1 conv
            x = Conv2D(2048, (1, 1), strides=(1, 1), padding="same", name=f"custom_conv5_block{i}_3_conv")(x)
            x = BatchNormalization(name=f"custom_conv5_block{i}_3_bn")(x)

            # Projection Shortcut
            shortcut = Conv2D(2048, (1, 1), strides=(1, 1), padding="same", name=f"custom_conv5_block{i}_0_conv")(shortcut)
            shortcut = BatchNormalization(name=f"custom_conv5_block{i}_0_bn")(shortcut)
        
        else:
            # 1x1 conv
            x = Conv2D(512, (1, 1), strides=(1, 1), padding="same", name=f"custom_conv5_block{i}_1_conv")(x)
            x = BatchNormalization(name=f"custom_conv5_block{i}_1_bn")(x)
            x = ReLU(name=f"custom_conv5_block{i}_1_relu")(x)
            
            # 3x3 atrous conv + dilation_rate=2
            x = Conv2D(512, (3, 3), strides=(1, 1), padding="same", dilation_rate=2, name=f"custom_conv5_block{i}_2_conv")(x)
            x = BatchNormalization(name=f"custom_conv5_block{i}_2_bn")(x)
            x = ReLU(name=f"custom_conv5_block{i}_2_relu")(x)

            # 1x1 conv
            x = Conv2D(2048, (1, 1), strides=(1, 1), padding="same", name=f"custom_conv5_block{i}_3_conv")(x)
            x = BatchNormalization(name=f"custom_conv5_block{i}_3_bn")(x)

        x = Add(name=f"custom_conv5_block{i}_add")([x, shortcut])
        x = ReLU(name=f"custom_conv5_block{i}_out")(x)

    encoder_output = x

    x = AtrousSpatialPyramidPooling(encoder_output)

    input_a = UpSampling2D(size=(4,4),interpolation="bilinear")(x)
    
    input_b = convolution_block(low_level_features, num_filters=48, kernel_size=1)
    
    x = Concatenate(axis=-1)([input_a, input_b])
    x = convolution_block(x)
    x = convolution_block(x)
    x = UpSampling2D(size=(4,4), interpolation="bilinear")(x)
    
    model_output = Conv2D(num_classes, kernel_size=(1, 1), padding="same")(x)
    
    return Model(inputs=model_input, outputs=model_output)

In [ ]:
model = DeeplabV3_OS16(image_size=IMG_SIZE[0], num_classes=4)

In [ ]:
model.summary()

In [ ]:
keras.utils.plot_model(model,show_layer_names=True)

In [ ]:
from tensorflow.keras import losses

In [ ]:
def dice_loss(y_true, y_pred, smooth=1e-6):

    y_pred_probs = tf.nn.softmax(y_pred, axis=-1)
    
    y_true_one_hot = tf.one_hot(tf.cast(y_true, tf.int32), depth=4, axis=-1)
    y_true_one_hot = tf.squeeze(y_true_one_hot, axis=-2)
    
    intersection = tf.reduce_sum(y_true_one_hot * y_pred_probs, axis=[1, 2])
    sum_true = tf.reduce_sum(y_true_one_hot, axis=[1, 2])
    sum_pred = tf.reduce_sum(y_pred_probs, axis=[1, 2])
    
    dice_coefficient = (2. * intersection ) / (sum_true + sum_pred + smooth)
    
    mean_dice_coefficient = tf.reduce_mean(dice_coefficient)
    
    return 1. - mean_dice_coefficient

def combined_loss(y_true, y_pred):
    
    scce = losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=True)
    dice = dice_loss(y_true, y_pred)
    
    return scce + dice

In [ ]:
sparse_mean_iou = tf.keras.metrics.MeanIoU(num_classes = 4, sparse_y_pred = False)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=combined_loss,
    metrics=["accuracy", sparse_mean_iou]
)

In [ ]:
from tensorflow.keras import callbacks
callbacks = [
    callbacks.ModelCheckpoint('./bestmodel.keras', 
                             monitor="val_loss",  
                             verbose=1,            
                             save_best_only=True),
]

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset, 
    epochs=20,
    class_weight=mfb_weights, 
    callbacks=callbacks
)

In [ ]:
plt.figure(figsize=(20, 5))
plt.subplot(1, 3, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(history.history['mean_io_u'], label='Training mean_iou')
plt.plot(history.history['val_mean_io_u'], label='val_mean_iou')
plt.title('mean_iou')
plt.legend()

plt.show()

# Evaluation

In [ ]:
custom_objects = {
    "combined_loss": combined_loss,
}


best_model = tf.keras.models.load_model(
    './bestmodel.keras',
    custom_objects=custom_objects
)

In [ ]:
def predict_and_visualize(model, image_path, mask_path, label, img_size=(256, 256), confidence_threshold=0.9):
   
    img = tf.io.read_file(image_path)
    img = tf.image.decode_png(img, channels=3)
    img_resized = tf.image.resize(img, img_size)
    
    img_for_display = img_resized 
    
    img_preprocessed = preprocess_input(img_resized)
    img_for_prediction = tf.expand_dims(img_preprocessed, axis=0)

    prediction = model.predict(img_for_prediction)

    max_probs = np.max(prediction, axis=-1)

    predicted_labels = np.argmax(prediction, axis=-1)

    predicted_mask = np.where(max_probs < confidence_threshold, 0, predicted_labels)

    predicted_mask = np.squeeze(predicted_mask)

    _, true_mask_tensor = load_image_and_mask(image_path, mask_path, label)
    true_mask = np.squeeze(true_mask_tensor.numpy())
    
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.title("Original Image")
    plt.imshow(img_for_display / 255.0)
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.title("True Mask")
    plt.imshow(true_mask, cmap='jet', vmin=0, vmax=3)
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.title(f"Predicted Mask (Threshold={confidence_threshold})")
    plt.imshow(predicted_mask, cmap='jet', vmin=0, vmax=3)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    print(f"Label: {label}")
    print(f"Unique values in True Mask: {np.unique(true_mask)}")
    print(f"Unique values in Predicted Mask: {np.unique(predicted_mask)}")

In [ ]:
sample_index = 1500 

sample_image_path = train_images[sample_index]
sample_mask_path = train_masks[sample_index]
sample_label = train_labels[sample_index]

predict_and_visualize(best_model, sample_image_path, sample_mask_path, sample_label)

In [ ]:
best_model.evaluate(train_dataset,return_dict=True,verbose=0)

In [ ]:
best_model.evaluate(val_dataset,return_dict=True,verbose=0)